In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 35.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [datasets]/15 [datasets]ce-hub]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
from datasets import load_dataset

# Load the Hindi subset of the dataset
dataset = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="hin_Deva", streaming=True)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import re
import json
from typing import List, Dict, Set, Generator, Any

class HindiTokenizer:
    def __init__(self):

        self.URL_PATTERN = r'(?:https?://)?(?:www\.)?[a-zA-Z0-9][a-zA-Z0-9-]*(?:\.[a-zA-Z0-9][a-zA-Z0-9-]*)+\.[a-zA-Z]{2,}'
        self.EMAIL_PATTERN = r'[\w\.-]+@[\w\.-]+\.\w+'
        self.DATE_PATTERN = r'(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4})|(?:\d{4}[/-]\d{1,2}[/-]\d{1,2})|(?:\d{1,2}\s+(?:जनवरी|फरवरी|मार्च|अप्रैल|मई|जून|जुलाई|अगस्त|सितम्बर|अक्टूबर|नवम्बर|दिसम्बर|January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{2,4})'
        self.NUMBER_PATTERN = r'\b\d+(?:\.\d+)?\b'
        self.HINDI_WORD_PATTERN = r'[\u0900-\u097F]+'
        self.ENGLISH_WORD_PATTERN = r'\b[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*\b'
        self.PUNCT_PATTERN = r'[^\s\w\u0900-\u097F]'
        self.SENTENCE_END_PATTERN = r'[।!?]+\s*'

        self.TOKEN_PATTERN = (
            f'({self.URL_PATTERN})|({self.EMAIL_PATTERN})|({self.DATE_PATTERN})|'
            f'({self.NUMBER_PATTERN})|({self.HINDI_WORD_PATTERN})|'
            f'({self.ENGLISH_WORD_PATTERN})|({self.PUNCT_PATTERN})'
        )

        self.token_re = re.compile(self.TOKEN_PATTERN)
        self.sentence_re = re.compile(self.SENTENCE_END_PATTERN)

        self.token_types = [
            'URL', 'EMAIL', 'DATE', 'NUMBER', 'HINDI_WORD', 'ENGLISH_WORD', 'PUNCTUATION'
        ]

    def sentence_tokenize(self, text: str) -> List[str]:
        text = text.strip()
        sentences = self.sentence_re.split(text)
        return [sent.strip() for sent in sentences if sent.strip()]

    def word_tokenize(self, text: str) -> List[Dict[str, Any]]:
        tokens = []
        for match in self.token_re.finditer(text):
            token_type = 'UNKNOWN'
            for i, group in enumerate(match.groups()):
                if group is not None:
                    token_type = self.token_types[i]
                    break
            tokens.append({
                'text': match.group(0), 'type': token_type,
            })
        return tokens

In [6]:
def large_dataset_generator() -> Generator[Dict[str, Any], None, None]:
    for idx,item in enumerate(dataset):
        if idx > 10000:
            return
        yield item

tokenizer = HindiTokenizer()
output_filename = 'tokenized_output.json'


total_docs_processed = 0
total_sentences = 0
total_words = 0
total_characters = 0
total_word_characters = 0

unique_words_for_ttr: Set[str] = set()

print(f"Processing dataset and streaming output to '{output_filename}'...")


with open(output_filename, 'w', encoding='utf-8') as f:
    for item in large_dataset_generator():
        paragraph = item.get('text')
        if not paragraph:
            continue

        paragraph_data = {
            'id': item.get('id', total_docs_processed),
            'original_text': paragraph,
            'sentences': []
        }

        sentences = tokenizer.sentence_tokenize(paragraph)

        total_characters += len(paragraph)
        total_sentences += len(sentences)

        for sent_idx, sentence_text in enumerate(sentences):
            tokens = tokenizer.word_tokenize(sentence_text)

            paragraph_data['sentences'].append({
                'sentence_id': sent_idx + 1, 'text': sentence_text, 'tokens': tokens
            })

            for token in tokens:
                if token['type'] in ['HINDI_WORD', 'ENGLISH_WORD']:
                    word_text = token['text'].lower()
                    total_words += 1
                    total_word_characters += len(word_text)
                    unique_words_for_ttr.add(word_text)

        f.write(json.dumps(paragraph_data, ensure_ascii=False) + '\n')

        total_docs_processed += 1

        if total_docs_processed % 1000 == 0:
            print(f"   ... processed {total_docs_processed} documents")

print(f"Processing complete. {total_docs_processed} documents processed.")
print(f"Tokenized data saved to '{output_filename}'")


total_unique_words = len(unique_words_for_ttr)


avg_sentence_length = total_words / total_sentences if total_sentences > 0 else 0
avg_word_length = total_word_characters / total_words if total_words > 0 else 0
type_token_ratio = total_unique_words / total_words if total_words > 0 else 0

print("\n" + "="*30)
print("FINAL CORPUS STATISTICS")
print("="*30)
print(f"i.   Total number of sentences    : {total_sentences}")
print(f"ii.  Total number of words        : {total_words} (Hindi/English words only)")
print(f"iii. Total number of characters    : {total_characters} (in original text)")
print(f"iv.  Average sentence length      : {avg_sentence_length:.2f} words/sentence")
print(f"v.   Average word length          : {avg_word_length:.2f} chars/word")
print(f"vi.  Type/Token Ratio (TTR)       : {type_token_ratio:.4f}")
print("="*30)

Processing dataset and streaming output to 'tokenized_output.json'...
   ... processed 1000 documents
   ... processed 2000 documents
   ... processed 3000 documents
   ... processed 4000 documents
   ... processed 5000 documents
Processing complete. 5001 documents processed.
Tokenized data saved to 'tokenized_output.json'

FINAL CORPUS STATISTICS
i.   Total number of sentences    : 14389
ii.  Total number of words        : 282275 (Hindi/English words only)
iii. Total number of characters    : 1453191 (in original text)
iv.  Average sentence length      : 19.62 words/sentence
v.   Average word length          : 4.00 chars/word
vi.  Type/Token Ratio (TTR)       : 0.0905
